# Validação de imagens – Modelo de integridade 

Autor:  Viviane da Rosa Sommer

Atualizado: 11/09/2025

Objetivo: Esse código tem como objetivo processar imagens em lote, classificando e separando-as em pastas de "válidas" ou "inválidas" com base no critério : não ser em escala de cinxa, FundoMar maior que 0.3 e Fundo Chão menor que 0.7. 

## Importações necessárias

In [ ]:
import numpy as np
import cv2
import glob
import os
import matplotlib.pyplot as plt
import shutil
import json
import tensorflow as tf
from tensorflow.keras.losses import BinaryCrossentropy
from tensorflow.keras.metrics import IoU
from tensorflow.keras.preprocessing import image
from keras.utils import get_custom_objects
import h5py
from utils.predict_functions import jaccard_distance
from PIL import Image, UnidentifiedImageError
import pandas as pd
import gc
from tqdm import tqdm

## Declaração de Constantes e Modelos

In [ ]:
get_custom_objects().update({'jaccard_distance': jaccard_distance})
model_name = 'pesos/modelcorrigido.h5'
classes_csv = 'pesos/training_classes.csv'

model = tf.keras.models.load_model(model_name, custom_objects={
    'loss': BinaryCrossentropy,
    'iou_score': IoU(num_classes=35, target_class_ids=[1] * 35)
})

with open(classes_csv, 'r') as f:
    class_names = f.readline().strip().split(',')

df = pd.read_csv('os_caminho_completo.csv')

pasta_principal = "originais"
batch_size = 12
extensoes = (".png", ".jpg", ".jpeg", ".bmp", ".tiff", ".gif")
pasta_saida = "fase1-0-7-3"

folders = [str(row['ID']) + "_" + str(row['OS']) for _, row in df.iterrows()]
folders = list(set(folders))

## Funções necessárias

In [ ]:
def load_and_preprocess(img_path: str, target_size: tuple[int, int] = (512, 512)) -> np.ndarray:
    """
    Loads an image, resizes it to target_size, and normalizes pixel values to [0, 1].

    Args:
        img_path (str): Path to the image file.
        target_size (tuple[int, int], optional): Desired image size. Defaults to (512, 512).

    Returns:
        np.ndarray: Preprocessed image array.
    """
    img = image.load_img(img_path, target_size=target_size)
    img_array = image.img_to_array(img)
    img_array = img_array / 255.0
    return img_array

def is_mostly_grayscale(img_path: str, threshold: float = 0.90, tolerance: int = 10) -> bool:
    """
    Checks if most pixels in the image are grayscale.

    Args:
        img_path (str): Path to the image file.
        threshold (float, optional): Minimum grayscale pixel ratio. Defaults to 0.90.
        tolerance (int, optional): Channel difference to consider as grayscale. Defaults to 10.

    Returns:
        bool: True if the image is mostly grayscale, False otherwise.
    """
    img = Image.open(img_path).convert('RGB')
    arr = np.array(img)
    diff_rg = np.abs(arr[..., 0] - arr[..., 1])
    diff_gb = np.abs(arr[..., 1] - arr[..., 2])
    diff_rb = np.abs(arr[..., 0] - arr[..., 2])
    mask = (diff_rg < tolerance) & (diff_gb < tolerance) & (diff_rb < tolerance)
    ratio = np.sum(mask) / mask.size
    return ratio > threshold

## Processamento das imagens em válido/inválido

In [ ]:
for folder in folders:
    caminho_folder = os.path.join(pasta_principal, folder)
    all_files = [os.path.join(root, f) for root, _, files in os.walk(caminho_folder) 
                 for f in files if f.lower().endswith(extensoes)]
    if not all_files:
        print(f"Nenhuma imagem encontrada na pasta {folder}")
        continue
    for i in tqdm(range(0, len(all_files), batch_size), desc=f"Processando {folder}"):
        batch_files = all_files[i:i+batch_size]
        batch_imgs, batch_paths = [], []
        for file_path in batch_files:
            rel_path = os.path.relpath(file_path, pasta_principal)
            dest_validas = os.path.join(pasta_saida, 'validas', rel_path)
            dest_invalidas = os.path.join(pasta_saida, 'invalidas', rel_path)
            if os.path.exists(dest_validas) or os.path.exists(dest_invalidas):
                continue
            try:
                if is_mostly_grayscale(file_path):
                    os.makedirs(os.path.dirname(dest_invalidas), exist_ok=True)
                    shutil.copy2(file_path, dest_invalidas)
                    continue
                img_array = load_and_preprocess(file_path)
                batch_imgs.append(img_array)
                batch_paths.append((file_path, dest_validas, dest_invalidas))
            except (UnidentifiedImageError, OSError):
                continue
        if batch_imgs:
            input_tensor = np.stack(batch_imgs, axis=0)
            preds = model.predict(input_tensor, verbose=0)
            for j, pred in enumerate(preds):
                file_path, dest_validas, dest_invalidas = batch_paths[j]
                duto = pred[class_names.index('Duto')]
                fundochao = pred[class_names.index('FundoChao')]
                fundomar = pred[class_names.index('FundoMar')]
                dest_path = dest_validas if fundochao < 0.7 and fundomar > 0.3 else dest_invalidas
                os.makedirs(os.path.dirname(dest_path), exist_ok=True)
                shutil.copy2(file_path, dest_path)
        gc.collect()
        tf.keras.backend.clear_session()
print("Processamento finalizado em batchs.")